In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-oss-20b:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

#### Discuss the Problem First

In [3]:
text = "Hello, My name is Ankit, Email is ankitabcd1718@gmail.com and my age is 21"

In [7]:
res = llm.invoke(f"Please give me only the name, email and age from the following text: {text}")

In [8]:
print(type(res.content))

<class 'str'>


#### Solution is Structured Output

In [17]:
from pydantic import BaseModel, Field
from typing import List

class ResponseStructure(BaseModel):
    name: str = Field(description="Complete Name")
    email: str = Field(description="Email Address")
    age: int = Field(description="Age of the person")


structured_llm = llm.with_structured_output(ResponseStructure)    

In [10]:
structured_res = structured_llm.invoke(f"Please give me only the name, email and age from the following text: {text}")

In [11]:
structured_res

ResponseStructure(name='Ankit', email='ankitabcd1718@gmail.com', age=21)

In [12]:
structured_res.name

'Ankit'

In [13]:
structured_res.model_dump()

{'name': 'Ankit', 'email': 'ankitabcd1718@gmail.com', 'age': 21}

In [14]:
class Movies(BaseModel):
    title: str = Field(description="Title of the movie")
    year: int = Field(description="Release year of the movie")
    genre: str = Field(description="Genre of the movie")
    director: str = Field(description="Director of the movie")
    cast: list[str] = Field(description="List of main cast members")
    rating: float = Field(description="Rating of the movie out of 10")

movie_llm = llm.with_structured_output(Movies)

In [15]:
movie_llm.invoke("Please provide the details of the movie 'Inception' including title, year, genre, director, cast, and rating.")

Movies(title='Inception', year=2010, genre='Science Fiction', director='Christopher Nolan', cast=['Leonardo DiCaprio', 'Joseph Gordon-Levitt', 'Elliot Page', 'Tom Hardy', 'Ken Watanabe', 'Cillian Murphy'], rating=8.8)

In [18]:
class AllMovies(BaseModel):
    movies: List[Movies] = Field(description="List of movies")

In [19]:
AllMovies_llm = llm.with_structured_output(AllMovies)

In [20]:
AllMovies_llm.invoke("Please provide the details of 3 movies including title, year, genre, director, cast, and rating.")

AllMovies(movies=[Movies(title='The Shawshank Redemption', year=1994, genre='Drama', director='Frank Darabont', cast=['Tim Robbins', 'Morgan Freeman', 'Bob Gunton'], rating=9.3), Movies(title='Inception', year=2010, genre='Sci-Fi', director='Christopher Nolan', cast=['Leonardo DiCaprio', 'Joseph Gordon-Levitt', 'Elliot Page'], rating=8.8), Movies(title='Parasite', year=2019, genre='Thriller', director='Bong Joon-ho', cast=['Song Kang-ho', 'Lee Sun-kyun', 'Cho Yeo-jeong'], rating=8.6)])